In [1]:
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
import pandas as pd
from langchain_chroma import Chroma
import torch
# print(torch.__version__, torch.version.cuda)
# print(torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [2]:
#importing dataset

df = pd.read_csv("../datasets/pubmed abstracts/pubmed_abstracts copy.csv")
df.head()

,Unnamed: 0,deep_learning
0,0,(['Magnetic resonance spectroscopic imaging (M...
1,1,(['Existing deep convolutional neural networks...
2,2,(['Deep learning techniques have been increasi...
3,3,(['The original article unfortunately containe...
4,4,(['The most common applications of artificial ...


In [3]:
#loading dataset in langchain

docs = []
for col in df.columns:
    if col == "Unnamed: 0":
        continue
    text = "\n".join(df[col].dropna().astype(str))
    docs.append(Document(page_content=text, metadata={"column": col}))

print(len(docs))   # = number of columns
print(docs[0].metadata)
print(type(docs[0].page_content))

1
{'column': 'deep_learning'}
<class 'str'>


In [4]:
#Spliting text

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
texts = text_splitter.split_documents(docs)
print(len(texts)) 
print(texts[5].page_content)

22842
by the classification branch. Feeding a chest radiograph to the trained Thorax-Net, a diagnosis is obtained by averaging and binarizing the outputs of two branches. The proposed Thorax-Net model has been evaluated against three state-of-the-art deep learning models using the patientwise official split of the ChestX-ray14 dataset and against other five deep learning models using the imagewise random data split. Our results show that Thorax-Net achieves an average per-class area under the receiver operating characteristic curve (AUC) of 0.7876 and 0.896 in both experiments, respectively, which are higher than the AUC values obtained by other deep models when they were all trained with no external data.'], 'Thorax-Net: An Attention Regularized Deep Neural Network for Classification of Thoracic Diseases on Chest Radiography.')


In [5]:
#Embeddings Model 

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"batch_size": 512},
    )

embeddings._client.half()                 # fp16
embeddings._client.max_seq_length = 256   # truncate long abstracts

query_embedding = embeddings.embed_query("What is a sentence embedding?")
doc_embeddings = embeddings.embed_documents(
    [
        "Sentence embeddings map text to dense vectors.",
        "LangChain provides a standard Embeddings interface.",
    ]
)

print(type(embeddings))
print(hasattr(embeddings, "_client"), hasattr(embeddings, "client"))

s:\Projects\rag-dissertation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 701.89it/s]


<class 'langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings'>
True False


In [7]:
m = embeddings._client
print(m.max_seq_length)                  # 256
print(next(m.parameters()).dtype)        # torch.float16
print(next(m.parameters()).device)       # cuda:0

256
torch.float16
cuda:0


In [ ]:
# Embeddings & vector storage
#
# This cell only READS the Chroma DB that was persisted to ./chroma_langchain_db
# (src/chroma_langchain_db) -- it does not re-embed the dataset on every run.

from pathlib import Path

CHROMA_DIR = Path("chroma_langchain_db")
if not CHROMA_DIR.exists():
    raise FileNotFoundError(
        f"Local Chroma DB not found at {CHROMA_DIR.resolve()}. "
        "Run the ingestion cells once with REBUILD_INDEX = True to create it."
    )

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
)

# Default LangChain retriever over the persisted collection.
retriever = vector_store.as_retriever()

print(f"Loaded {vector_store._collection.count()} chunks from {CHROMA_DIR.resolve()}")


In [ ]:
# Timeline of storage
#
# The index already exists on disk, so ingestion is opt-in: flip REBUILD_INDEX to
# True only when you want to (re)embed the split texts into the collection.

from tqdm import tqdm

REBUILD_INDEX = False

if REBUILD_INDEX:
    max_bs = vector_store._client.get_max_batch_size()   # 5461 for you
    BATCH = min(2000, max_bs)                            # stay safely under it

    for i in tqdm(range(0, len(texts), BATCH)):
        batch = texts[i:i + BATCH]
        ids = [str(d.metadata.get("pmid", i + j)) for j, d in enumerate(batch)]
        vector_store.add_documents(batch, ids=ids)
else:
    print("Skipping ingestion; reading the existing local Chroma DB as-is.")


In [16]:
# Retrieve evidence for the same question that will be evaluated.
query = "Which NN do we mainly use for ADHD Classification?"
retrieved_docs = retriever.invoke(query)
retrieval_context = [doc.page_content for doc in retrieved_docs]
if not retrieval_context:
    raise ValueError("The retriever returned no context; cannot evaluate a RAG answer.")
print(f"Retrieved {len(retrieval_context)} context chunks.")


Retrieved 4 context chunks.


In [13]:
#Setting up LLM

llm = ChatOllama(
    model="qwen3.5:2b", 
    temperature = 0.5,
    reasoning=False,
    
)

In [17]:
# Generate the RAG answer from the retrieved evidence.
context_text = "\n\n".join(retrieval_context)
prompt = f"""Answer the question using only the supplied context. If the context does not contain the answer, say so.

Context:
{context_text}

Question: {query}
Answer:"""
response = llm.invoke(prompt)
print(response.content)


Based on the context provided, specifically the sentence "DeepFMRI: End-to-end deep learning for functional connectivity and classification of ADHD using fMRI," the neural network mentioned is **DeepFMRI**.

(Note: The text also mentions an "EEGNet" in a different study regarding EEG data, but DeepFMRI is the model explicitly named in the context provided for classification using resting-state functional connectivity.)


In [ ]:
# Sanity check the local Ollama server with the judge model.
import ollama

r = ollama.chat(
    model="qwen3.5:2b",
    messages=[{"role": "user", "content": 'Say hi in JSON: {"msg": ...}'}],
)
print(repr(r.message.content))


In [19]:
import os

# The local judge can be slow, so allow longer local judge calls.
os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "300"
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "600"

actual_context =[
    "Attention Deficit Hyperactivity Disorder (ADHD) is one of the most prevalent neuropsychiatric disorders in childhood and adolescence and its diagnosis is based on clinical interviews, symptom questionnaires, and neuropsychological testing. Much research effort has been undertaken to evaluate the usefulness of neurophysiological (EEG) data to aid this diagnostic process. In the current study, we applied deep learning methods on event-related EEG data to examine whether it is possible to distinguish ADHD patients from healthy controls using purely neurophysiological measures. The same was done to distinguish between ADHD subtypes. The results show that the applied deep learning model (\"EEGNet\") was able to distinguish between both ADHD subtypes and healthy controls with an accuracy of up to 83%. However, a significant fraction of individuals could not be classified correctly. It is shown that neurophysiological processes indicating attentional selection associated with superior parietal cortical areas were the most important for that. Using the applied deep learning method, it was not possible to distinguish ADHD subtypes from each other. This is the first study showing that deep learning methods applied to EEG data are able to dissociate between ADHD patients and healthy controls. The results show that the applied method reflects a promising means to support clinical diagnosis in ADHD. However, more work needs to be done to increase the reliability of the taken approach.'], 'Deep Learning Based on Event-Related EEG Differentiates Children with ADHD from Healthy Controls.",
    "Attention deficit/hyperactivity disorder(ADHD) is a behavioral disorder syndrome found mainly in school-age population. At present, the diagnosis of ADHD mainly depends on the subjective methods, leading to the high rate of misdiagnosis and missed-diagnosis. To solve these problems, we proposed an algorithm for classifying ADHD objectively based on convolutional neural network. At first, preprocessing steps, including skull stripping, Gaussian kernel smoothing, et al., were applied to brain magnetic resonance imaging(MRI). Then, coarse segmentation was used for selecting the right caudate nucleus, left precuneus, and left superior frontal gyrus region. Finally, a 3 level convolutional neural network was used for classification. Experimental results showed that the proposed algorithm was capable of classifying ADHD and normal groups effectively, the classification accuracies obtained by the right caudate nucleus and the left precuneus brain regions were greater than the highest classification accuracy(62.52%) in the ADHD-200 competition, and among 3 brain regions in ADHD and the normal groups, the classification accuracy from the right caudate nucleus was the highest. It is well concluded that the method for classification of ADHD and normal groups proposed in this paper utilizing the coarse segmentation and deep learning is a useful method for the purpose. The classification accuracy of the proposed method is high, and the calculation is simple. And the method is able to extract the unobvious image features better, and can overcome the shortcomings of traditional methods of MRI brain area segmentation, which are time-consuming and highly complicate. The method provides an objective diagnosis approach for ADHD.'], '[Study of attention deficit/hyperactivity disorder classification based on convolutional neural networks",
    "Attention deficit and hyperactivity disorder (ADHD) is a neurodevelopmental disorder, which is characterized by inattention, hyperactivity and impulsive behaviors. In particular, children have difficulty keeping still exhibiting increased fine and gross motor activity. This paper focuses on analyzing the data obtained from two tri-axial accelerometers (one on the wrist of the dominant arm and the other on the ankle of the dominant leg) worn during school hours by a group of 22 children (11 children with ADHD and 11 paired controls). Five of the 11 ADHD diagnosed children were not on medication during the study. The children were not explicitly instructed to perform any particular activity but followed a normal session at school alternating classes of little or moderate physical activity with intermediate breaks of more prominent physical activity. The tri-axial acceleration signals were converted into 2D acceleration images and a Convolutional Neural Network (CNN) was trained to recognize the differences between non-medicated ADHD children and their paired controls. The results show that there were statistically significant differences in the way the two groups moved for the wrist accelerometer (t-test p-value <0.05). For the ankle accelerometer statistical significance was only achieved between data from the non-medicated children in the experimental group and the control group. Using a Convolutional Neural Network (CNN) to automatically extract embedded acceleration patterns and provide an objective measure to help in the diagnosis of ADHD, an accuracy of 0.875 for the wrist sensor and an accuracy of 0.9375 for the ankle sensor was achieved.'], 'Automatic Extraction and Detection of Characteristic Movement Patterns in Children with ADHD Based on a Convolutional Neural Network (CNN) and Acceleration Images.",
]

from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    HallucinationMetric,
)
from deepeval.models.llms.ollama_model import OllamaModel, retry_ollama
from deepeval.test_case import Audio, LLMTestCase
from pydantic import BaseModel
from typing import List, Optional, Tuple, Union


class NonThinkingOllamaModel(OllamaModel):
    @retry_ollama
    def generate(
        self,
        prompt: str,
        schema: Optional[BaseModel] = None,
        audio: Optional[List[Audio]] = None,
    ) -> Tuple[Union[str, BaseModel], float]:
        response = self.load_model().chat(
            model=self.name,
            messages=self._prompt_messages(prompt, audio),
            think=False,
            format=schema.model_json_schema() if schema else None,
            options={
                "temperature": self.temperature,
                **self.generation_kwargs,
            },
        )
        content = response.message.content
        if not content:
            raise ValueError(
                "Ollama returned empty content for the DeepEval judge response."
            )
        result = schema.model_validate_json(content) if schema else content
        return result, 0

# Use a separate local judge; Qwen 3.5 otherwise returns reasoning instead of JSON.
evaluation_model = NonThinkingOllamaModel(
    model="qwen3.5:2b",
    base_url="http://localhost:11434",
    temperature=0,
)

test_case = LLMTestCase(
    input=query,
    actual_output=response.content,
    context=actual_context,
    retrieval_context=retrieval_context,
)

metrics = {
    "hallucination": HallucinationMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
    "faithfulness": FaithfulnessMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
    "answer relevancy": AnswerRelevancyMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
}

for name, metric in metrics.items():
    metric.measure(test_case)
    print(f"{name}: {metric.score:.3f}")
    print(f"  passed: {metric.is_successful()}")
    print(f"  reason: {metric.reason}")


s:\Projects\rag-dissertation\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

hallucination: 0.000
  passed: False
  reason: The hallucination score is 1.0 because the actual output incorrectly identifies 'EEGNet' as 'DeepFMRI' and misattributes the use of fMRI to a model that was explicitly described in the context using EEG data, directly contradicting the provided factual evidence.


faithfulness: 0.333
  passed: False
  reason: The actual output incorrectly claims DeepFMRI is a neural network and uses fMRI for ADHD classification, contradicting the retrieval context which specifies that DeepFMRI is an EEG-based method used for functional connectivity and ADHD classification.


answer relevancy: 0.500
  passed: True
  reason: The score is 0.50 because the output contains two distinct irrelevant statements: one incorrectly identifies a model as DeepFMRI (fMRI-related) and another discusses EEGNet (EEG-related), neither of which addresses the specific question about the neural network for ADHD classification.
